# Pipeline medalhão — Superstore Sales
Construção do pipeline de dados seguindo a arquitetura medalhão:
1. **Bronze**: ingestão bruta do arquivo `train.csv`
2. **Silver**: padronização de nomes e tipos
3. **Gold**: modelo dimensional em esquema estrela + consultas analíticas 

## Definindo variáveis
Variáveis reutilizadas nas células seguintes: catálogo, caminho do volume com os arquivos de origem e nomes das tabelas bronze e silver.

In [0]:

CATALOG = "superstore_sales"
volume_path = "/Volumes/superstore_sales/bronze/external_files/"
bronze_table = "bronze.train"
silver_table = "silver.sales"

## 1. Camada bronze
Ingestão do `train.csv` via `read_files`, sem tratamentos. São adicionadas colunas de metadados (caminho do arquivo e timestamp de ingestão) e os nomes de colunas com espaços são substituídos por underscores.

**Resultado:** `bronze.train` 

In [0]:

# Lê o CSV usando read_files e adiciona colunas de metadados
df = spark.sql(f"""
SELECT
     *
    ,_metadata.file_path AS _ingestion_file_path
    ,current_timestamp() AS _ingestion_ts
FROM read_files("{volume_path}", format => 'csv', header => true, encoding => 'ISO-8859-1')
""")

# Renomeia colunas substituindo espaços por underscores
new_columns = [c.replace(" ", "_") for c in df.columns]
df = df.toDF(*new_columns)

# Cria a tabela bronze
df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{bronze_table}")
print(f"Tabela {CATALOG}.{bronze_table} criada com {df.count()} registros.")

## 2. Camada silver
Padronização da bronze: nomes em *snake_case*, conversão de datas e valores com `TRY_CAST` e exclusão de registros com `sales` nulo, inválido ou menor/igual a zero.

**Resultado:** `silver.sales`

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{silver_table} 
SELECT 
     Row_ID AS row_id
    ,Order_ID AS order_id
    ,TRY_CAST(Order_Date AS date) AS order_date
    ,TRY_CAST(Ship_Date AS date) AS ship_date
    ,Ship_Mode AS ship_mode
    ,Customer_ID AS custumer_id
    ,Customer_Name AS custumer_name
    ,Segment AS segment
    ,Country AS country
    ,City AS city
    ,State AS state
    ,Postal_Code AS postal_code
    ,Region AS region
    ,Product_ID AS product_id
    ,Category AS category
    ,`Sub-Category` AS subcategory
    ,Product_Name AS product_name
    ,TRY_CAST(Sales AS decimal(18,2)) AS sales
FROM {CATALOG}.{bronze_table}
WHERE TRY_CAST(Sales AS decimal(18,2)) > 0
""")

## 3. Camada gold
Modelo dimensional em esquema estrela: dimensões (`dim_date`, `dim_custumer`, `dim_product`, `dim_city`) e a tabela fato (`fato_sale_item`), com *surrogate keys* geradas automaticamente e *constraints* de integridade (PK, UNIQUE e FK).

### 3.1 Dimensão data
Calendário com uma linha por dia, gerado da menor à maior data de pedido em `silver.sales`. A chave `id_date` (formato `yyyymmdd`) é usada como FK nas colunas de data da tabela fato.

In [0]:
%sql
DROP TABLE IF EXISTS superstore_sales.gold.dim_date;
CREATE TABLE superstore_sales.gold.dim_date
    (id_date INT PRIMARY KEY COMMENT 'Id da data no formato yyyymmdd (ex: 20261016)'
    ,complete_date TIMESTAMP COMMENT 'Data completa'
    ,day INT COMMENT 'Dia do mes (1-31)'
    ,week_day STRING COMMENT 'Nome do dia da semana'
    ,month INT COMMENT 'Numero do mes (1-12)'
    ,month_name STRING COMMENT 'Nome do mes'
    ,year INT COMMENT 'Ano (ex: 2026)');

INSERT INTO superstore_sales.gold.dim_date (id_date, complete_date, day, week_day, month, month_name, year)
SELECT
     CAST(DATE_FORMAT(d.dt, 'yyyyMMdd') AS INT) AS id_date
    ,CAST(d.dt AS TIMESTAMP)                    AS data_completa
    ,DAY(d.dt)                                   AS dia
    ,DATE_FORMAT(d.dt, 'EEEE')                   AS dia_semana
    ,MONTH(d.dt)                                 AS mes
    ,DATE_FORMAT(d.dt, 'MMMM')                   AS mes_nome
    ,YEAR(d.dt)                                  AS ano
FROM (
    SELECT explode(sequence(
         (SELECT CAST(min(order_date) AS DATE) FROM superstore_sales.silver.sales)
        ,(SELECT CAST(max(order_date) AS DATE) FROM superstore_sales.silver.sales)
        ,INTERVAL 1 DAY
    )) AS dt
) d


### 3.2 Dimensão Custumer
Uma linha por cliente (`custumer_id`), com *surrogate key* gerada. Nomes nulos preenchidos com 'NÃO IDENTIFICADO'.

In [0]:
%sql
DROP TABLE IF EXISTS superstore_sales.gold.dim_custumer;
CREATE TABLE superstore_sales.gold.dim_custumer
    (sk_custumer BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) PRIMARY KEY COMMENT 'Surrogate key auto-gerada para a dimensao de cliente (chave primaria)'
    ,custumer_id STRING COMMENT 'Chave de negocio referente ao identificador do cliente'
    ,custumer_name STRING COMMENT 'Nome do cliente');

INSERT INTO superstore_sales.gold.dim_custumer (custumer_id, custumer_name)
SELECT 
     custumer_id
    ,coalesce(custumer_name, 'NÃO IDENTIFICADO')
FROM superstore_sales.silver.sales
WHERE custumer_id IS NOT NULL
GROUP BY custumer_id, custumer_name

### 3.3 Dimensão produto
Uma linha por produto (`product_id`) com nome, categoria e subcategoria. Quando o mesmo `product_id` tiver atributos divergentes, é mantida a primeira ocorrência (menor `row_id`) para respeitar a *constraint* `UNIQUE`.

In [0]:
%sql
DROP TABLE IF EXISTS superstore_sales.gold.dim_product;
CREATE TABLE superstore_sales.gold.dim_product
    (sk_product BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) PRIMARY KEY COMMENT 'Surrogate key auto-gerada para a dimensao de produto (chave primaria)'
    ,product_id STRING COMMENT 'Chave de negocio referente ao identificador do produto'
    ,product_name STRING COMMENT 'Nome do produto'
    ,category STRING COMMENT 'Categoria do produto'
    ,subcategory STRING COMMENT 'Sub-Categoria do produto');

INSERT INTO superstore_sales.gold.dim_product (product_id, product_name, category, subcategory)
SELECT 
     product_id
    ,product_name
    ,category
    ,subcategory
FROM superstore_sales.silver.sales
WHERE product_id IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY row_id) = 1

### 3.4 Dimensão city
Uma linha por combinação de cidade, estado e região. O país foi descartado pois todos os registros são dos Estados Unidos.

In [0]:
%sql
DROP TABLE IF EXISTS superstore_sales.gold.dim_city;
CREATE TABLE superstore_sales.gold.dim_city
    (sk_city BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) PRIMARY KEY COMMENT 'Surrogate key auto-gerada para a dimensao de cidade (chave primaria)'
    ,city STRING COMMENT 'Nome da cidade'
    ,state STRING COMMENT 'Estado'
    ,region STRING COMMENT 'Regiao');

INSERT INTO superstore_sales.gold.dim_city (city, state, region)
SELECT 
    city
    ,state
    ,region
FROM superstore_sales.silver.sales
GROUP BY city, state, region

### 3.5 Fato item de venda
**Granularidade:** item de venda. Tabela fato com a métrica `sales` e as FKs para as dimensões (datas de pedido e envio, cliente, produto e cidade). O `sale_item_id` é composto por `order_id` + sequencial do item no pedido.

In [0]:
%sql
DROP TABLE IF EXISTS superstore_sales.gold.fato_sale_item;
CREATE TABLE superstore_sales.gold.fato_sale_item 
    (sk_sale_item BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) PRIMARY KEY COMMENT 'Surrogate key auto-gerada para a fato de item de venda (chave primaria)'
    ,sale_item_id STRING COMMENT 'Chave de identificação de item de venda. Junção de (order_id) + (sequencial calculado)'
    ,order_id STRING COMMENT 'Identificador do pedido'
    ,id_date_order INT COMMENT 'Surrogate key referente a data do pedido (FK para dim_date)'
    ,id_date_ship INT COMMENT 'Surrogate key referente a data de envio (FK para dim_date)'
    ,sk_custumer BIGINT COMMENT 'Surrogate key referente ao cliente (FK para dim_custumer)'
    ,sk_product BIGINT COMMENT 'Surrogate key referente ao produto (FK para dim_product)'
    ,sk_city BIGINT COMMENT 'Surrogate key referente a cidade (FK para dim_city)'
    ,ship_mode STRING COMMENT 'Modo de envio'
    ,segment STRING COMMENT 'Segmento do cliente'
    ,postal_code STRING COMMENT 'Codigo postal'
    ,sales DECIMAL(18,2) COMMENT 'Valor da venda (metrica)'
    ,CONSTRAINT fk_custumer FOREIGN KEY (sk_custumer) REFERENCES superstore_sales.gold.dim_custumer (sk_custumer)
    ,CONSTRAINT fk_product FOREIGN KEY (sk_product) REFERENCES superstore_sales.gold.dim_product (sk_product)
    ,CONSTRAINT fk_city FOREIGN KEY (sk_city) REFERENCES superstore_sales.gold.dim_city (sk_city)
    ,CONSTRAINT fk_date_order FOREIGN KEY (id_date_order) REFERENCES superstore_sales.gold.dim_date (id_date)
    ,CONSTRAINT fk_date_ship FOREIGN KEY (id_date_ship) REFERENCES superstore_sales.gold.dim_date (id_date));

INSERT INTO superstore_sales.gold.fato_sale_item
    (sale_item_id, order_id, id_date_order, id_date_ship, sk_custumer, sk_product, sk_city, ship_mode, segment, postal_code, sales)
SELECT
     CONCAT(s.order_id,'_',RANK() OVER (PARTITION BY s.order_id ORDER BY s.row_id)) AS sale_item_id
    ,s.order_id
    ,date_format(s.order_date, 'yyyyMMdd') AS id_date_order
    ,date_format(s.ship_date, 'yyyyMMdd') AS id_date_ship
    ,dc.sk_custumer AS sk_custumer
    ,dp.sk_product  AS sk_product
    ,dci.sk_city    AS sk_city
    ,s.ship_mode
    ,s.segment
    ,s.postal_code
    ,s.sales
FROM superstore_sales.silver.sales s
LEFT JOIN superstore_sales.gold.dim_custumer dc
    ON dc.custumer_id = s.custumer_id
LEFT JOIN superstore_sales.gold.dim_product dp
    ON dp.product_id = s.product_id
LEFT JOIN superstore_sales.gold.dim_city dci
    ON dci.city = s.city AND dci.state = s.state AND dci.region = s.region

In [0]:
display(_sqldf)

## 4. Consultas
Análises de negócio sobre a camada gold, explorando o modelo dimensional.

### 4.1 Somas de vendas por mês em cada região
Soma de `sales` por mês e região, via junção da fato com `dim_city` e `dim_date`.

In [0]:
%sql
SELECT 
     C.region
    ,D.month_name
    ,SUM(SI.sales) agg_sales
FROM superstore_sales.gold.fato_sale_item SI
    LEFT JOIN superstore_sales.gold.dim_city C ON C.sk_city = SI.sk_city
    LEFT JOIN superstore_sales.gold.dim_date D ON D.id_date = SI.id_date_order
GROUP BY C.region, D.month_name, D.month
ORDER BY C.region, D.month

### 4.2 TOP 10 estados com mais pedidos
Ranking dos estados por quantidade de pedidos distintos, usando `RANK`.

In [0]:
%sql
WITH ranking_uf AS (
    SELECT
         c.state
        ,COUNT(DISTINCT si.order_id) AS qtd_pedidos
        ,RANK() OVER (ORDER BY COUNT(DISTINCT si.order_id) DESC) AS ranking
    FROM superstore_sales.gold.fato_sale_item si
    INNER JOIN superstore_sales.gold.dim_city c
        ON si.sk_city = c.sk_city
    GROUP BY c.state
)
SELECT
     ranking
    ,state
    ,qtd_pedidos
FROM ranking_uf
WHERE ranking <= 10
ORDER BY ranking

### 4.3 Crescimento mensal de vendas de 2015
Vendas mensais de 2015 comparadas ao mês anterior com `LAG`.

In [0]:
%sql
WITH sales_by_month AS
(
    SELECT 
         D.month
        ,D.month_name
        ,SUM(SI.sales) AS agg_sales
    FROM superstore_sales.gold.fato_sale_item SI
        LEFT JOIN superstore_sales.gold.dim_date D ON D.id_date = SI.id_date_order
    WHERE D.year = 2015
    GROUP BY D.month, D.month_name
)
SELECT
     month_name mes
    ,LAG(agg_sales) OVER (ORDER BY month) AS vendas_mes_anterior
    ,agg_sales as vendas_atual
FROM sales_by_month
ORDER BY month